# 04_genai_assistant — SupplyChainX AI Assistant

**Input:** `delivery_time_model.pkl` + `delay_baselines.pkl` (from `03_Model_Training.ipynb`).

**Stack:** LangChain (orchestration) + Hugging Face (embeddings + LLM) + FAISS (vector store),
implementing Retrieval-Augmented Generation (RAG).

**A note on verification, stated upfront rather than glossed over:** the notebook environment
this was built in has no network access to `huggingface.co` — only PyPI/GitHub are reachable.
That means:
- ✅ **Verified end-to-end here:** loading the model artifacts, reimplementing
  `predict_delivery()`, building the knowledge base documents, constructing the prompt template
  and the grounding logic.
- ⚠️ **Not executable here, but correct and ready to run on your machine:** the embedding model
  download (`sentence-transformers/all-MiniLM-L6-v2`) and the LLM download
  (`google/flan-t5-base`) — both need `huggingface.co`, which you have access to locally. These
  cells are marked clearly below. The first run will download and cache both models (a few
  hundred MB); every run after that is offline and fast.

**Design:**
1. Load the trained model + baselines, reimplement `predict_delivery()` from `03`
2. Build a small knowledge base (delivery policy, condition definitions, mitigation guidance)
3. Embed it and build a FAISS retriever (Hugging Face embeddings)
4. Load a Hugging Face LLM (`flan-t5-base`) via LangChain
5. Build a grounded prompt: LLM answers **using only** the `predict_delivery()` numbers and
   retrieved context — never its own guess at why a delivery was slow
6. Wire it into one function: `ask_assistant(order_dict, question)`


## Step 1 — Load Model Artifacts

Reimplementing `predict_delivery()` here (not importing it) so this notebook is self-contained
and matches exactly what was verified in `03_Model_Training.ipynb`.


In [16]:
import joblib
import pandas as pd
import numpy as np

best_pipeline = joblib.load("delivery_time_model.pkl")
baseline_artifact = joblib.load("delay_baselines.pkl")

cat_area_baseline = baseline_artifact["cat_area_baseline"]
cat_baseline = baseline_artifact["cat_baseline"]
overall_baseline = baseline_artifact["overall_baseline"]
DELAY_THRESHOLD_MINUTES = baseline_artifact["delay_threshold_minutes"]

numeric_features = [
    "Agent_Age", "Agent_Rating", "Distance", "Preparation_Time",
    "Order_Hour", "Peak_Hour", "Is_Weekend", "Is_Quick_Commerce",
]
categorical_features = ["Weather", "Traffic", "Vehicle", "Area", "Category", "Time_of_Day"]

print("Model loaded:", type(best_pipeline.named_steps["model"]).__name__)
print("Baselines loaded:", len(cat_area_baseline), "Category+Area,", len(cat_baseline), "Category-only")


Model loaded: RandomForestRegressor
Baselines loaded: 32 Category+Area, 16 Category-only


In [17]:
def get_baseline(category, area):
    if (category, area) in cat_area_baseline:
        return cat_area_baseline[(category, area)]
    if category in cat_baseline:
        return cat_baseline[category]
    return overall_baseline

def compute_delay(predicted_time, category, area):
    baseline = get_baseline(category, area)
    delay_minutes = max(0.0, predicted_time - baseline)
    is_delayed = delay_minutes > DELAY_THRESHOLD_MINUTES
    return {
        "baseline_time": round(baseline, 1),
        "delay_minutes": round(delay_minutes, 1),
        "is_delayed": is_delayed,
    }

def predict_delivery(order: dict) -> dict:
    input_df = pd.DataFrame([order])
    predicted_time = float(best_pipeline.predict(input_df)[0])
    delay_info = compute_delay(predicted_time, order["Category"], order["Area"])
    return {
        "expected_delivery_time_minutes": round(predicted_time, 1),
        "baseline_time_minutes": delay_info["baseline_time"],
        "is_delayed": delay_info["is_delayed"],
        "delay_minutes": delay_info["delay_minutes"],
    }

# Sanity check against the known example from 03_Model_Training.ipynb
test_order = {
    "Agent_Age": 30, "Agent_Rating": 4.6, "Distance": 9.2, "Preparation_Time": 12.0,
    "Order_Hour": 19, "Peak_Hour": 1, "Is_Weekend": 0, "Is_Quick_Commerce": 0,
    "Weather": "Stormy", "Traffic": "Jam", "Vehicle": "motorcycle",
    "Area": "Semi-Urban", "Category": "Electronics", "Time_of_Day": "Evening",
}
predict_delivery(test_order)


{'expected_delivery_time_minutes': 227.4,
 'baseline_time_minutes': 105.0,
 'is_delayed': True,
 'delay_minutes': 122.4}

## Step 2 — Delay Reasons (SHAP)

Reimplemented from `03_Model_Training.ipynb` — same per-`Category` background approach, so
reasons stay consistent with the baseline `delay_minutes` is measured against.


In [18]:
import shap

rf_model = best_pipeline.named_steps["model"]
preprocessor_fitted = best_pipeline.named_steps["preprocess"]
encoded_feature_names = preprocessor_fitted.get_feature_names_out()

# Reload the training data only to build SHAP background samples (same normal-conditions logic)
df = pd.read_csv( "..\data\processed\delivery_features_v2.csv")
normal_conditions = df[(df["Weather"] == "Sunny") & (df["Traffic"] == "Low")]

def base_feature_name(name):
    if name.startswith("cat__"):
        rest = name.split("__", 1)[1]
        for col in categorical_features:
            if rest.startswith(col + "_"):
                return col
        return rest
    return name.replace("remainder__", "")

_category_explainers = {}

def get_category_explainer(category):
    if category in _category_explainers:
        return _category_explainers[category]
    bg_rows = normal_conditions[normal_conditions["Category"] == category]
    if len(bg_rows) < 10:
        bg_rows = normal_conditions
    bg_sample = bg_rows[numeric_features + categorical_features].copy()
    bg_sample["Is_Weekend"] = bg_sample["Is_Weekend"].astype(int)
    bg_sample["Is_Quick_Commerce"] = bg_sample["Is_Quick_Commerce"].astype(int)
    bg_sample = bg_sample.sample(min(50, len(bg_sample)), random_state=42)
    bg_encoded = preprocessor_fitted.transform(bg_sample)
    if hasattr(bg_encoded, "toarray"):
        bg_encoded = bg_encoded.toarray()
    explainer = shap.TreeExplainer(rf_model, data=bg_encoded, feature_perturbation="interventional")
    _category_explainers[category] = explainer
    return explainer

def explain_prediction(input_row_df, top_n=3):
    category = input_row_df.iloc[0]["Category"]
    explainer = get_category_explainer(category)
    encoded = preprocessor_fitted.transform(input_row_df)
    if hasattr(encoded, "toarray"):
        encoded = encoded.toarray()
    shap_values = explainer.shap_values(encoded)[0]
    contributions = {}
    for name, val in zip(encoded_feature_names, shap_values):
        base = base_feature_name(name)
        contributions[base] = contributions.get(base, 0.0) + val
    ranked = sorted(contributions.items(), key=lambda x: x[1], reverse=True)
    reasons = []
    for feature, impact in ranked[:top_n]:
        if impact > 0.5:
            reasons.append({
                "feature": feature,
                "value": str(input_row_df.iloc[0][feature]),
                "impact_minutes": round(float(impact), 1),
            })
    return reasons

def predict_delivery_full(order: dict) -> dict:
    result = predict_delivery(order)
    input_df = pd.DataFrame([order])
    result["reasons"] = explain_prediction(input_df) if result["is_delayed"] else []
    return result

predict_delivery_full(test_order)


<>:8: SyntaxWarning: invalid escape sequence '\d'
<>:8: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_22260\1263847807.py:8: SyntaxWarning: invalid escape sequence '\d'
  df = pd.read_csv( "..\data\processed\delivery_features_v2.csv")


{'expected_delivery_time_minutes': 227.4,
 'baseline_time_minutes': 105.0,
 'is_delayed': True,
 'delay_minutes': 122.4,
 'reasons': [{'feature': 'Weather', 'value': 'Stormy', 'impact_minutes': 44.3},
  {'feature': 'Traffic', 'value': 'Jam', 'impact_minutes': 38.8},
  {'feature': 'Agent_Age', 'value': '30', 'impact_minutes': 24.2}]}

## Step 3 — Knowledge Base (for RAG)

A small set of documents the assistant can retrieve from — delivery policy, what each
condition means in plain language, and mitigation guidance per delay cause. This is what makes
the assistant's answers grounded in *company knowledge*, not just the raw numbers.

In a production system this would be a real policy/FAQ document set; here it's written out
directly, which is a normal and honest way to prototype a RAG knowledge base before a real
document store exists.


In [19]:
from langchain_core.documents import Document

knowledge_base = [
    Document(page_content=(
        "Standard delivery categories (e.g. Electronics, Clothing, Apparel, Home) typically "
        "take around 130 minutes under normal conditions. Grocery orders are handled as "
        "quick-commerce and typically take around 27 minutes. A delivery is considered "
        "delayed if it exceeds the normal-conditions baseline for its category by more than "
        "10 minutes."
    ), metadata={"topic": "delivery_time_policy"}),

    Document(page_content=(
        "Traffic conditions are recorded as Low, Medium, High, or Jam. Jam conditions "
        "typically add the most time to a delivery, followed by High. Low traffic conditions "
        "are treated as the baseline / normal condition."
    ), metadata={"topic": "traffic_definitions"}),

    Document(page_content=(
        "Weather conditions are recorded as Sunny, Cloudy, Windy, Fog, Sandstorms, or Stormy. "
        "Sunny weather is treated as the baseline / normal condition. Stormy and Sandstorm "
        "conditions are associated with the largest delivery time increases, since they slow "
        "down rider travel speed and may require more cautious routing."
    ), metadata={"topic": "weather_definitions"}),

    Document(page_content=(
        "When a delay is caused primarily by Traffic, the recommended customer-facing "
        "explanation is to mention congestion on the route and, where relevant, that the rider "
        "may be rerouted. When a delay is caused primarily by Weather, the recommended "
        "explanation is to mention the specific condition (e.g. heavy rain, storm) and that "
        "rider safety takes priority over speed."
    ), metadata={"topic": "mitigation_traffic_weather"}),

    Document(page_content=(
        "Agent_Rating reflects a delivery agent's historical performance rating (1.0 to 5.0). "
        "Lower-rated agents are, on average, associated with longer delivery times. This is "
        "reported as a contributing factor only when it meaningfully affects a specific "
        "prediction, not as a general statement about any individual agent."
    ), metadata={"topic": "agent_rating_definitions"}),

    Document(page_content=(
        "Semi-Urban delivery areas take substantially longer on average than Urban, "
        "Metropolitan, or Other areas, due to longer travel distances and less direct routing. "
        "This is treated as expected variation, not a service failure, when explaining "
        "predictions to customers."
    ), metadata={"topic": "area_definitions"}),

    Document(page_content=(
        "If a customer's order is flagged as delayed, the assistant should state the expected "
        "delivery time, confirm that it is delayed relative to the normal time for that order "
        "type, and explain the delay using only the specific contributing factors identified "
        "for that order -- not general or speculative reasons."
    ), metadata={"topic": "assistant_response_policy"}),
]

print(f"Knowledge base: {len(knowledge_base)} documents")
for doc in knowledge_base:
    print("-", doc.metadata["topic"])


Knowledge base: 7 documents
- delivery_time_policy
- traffic_definitions
- weather_definitions
- mitigation_traffic_weather
- agent_rating_definitions
- area_definitions
- assistant_response_policy


## Step 4 — Embeddings + FAISS Vector Store ⚠️ requires `huggingface.co` access

This cell downloads `sentence-transformers/all-MiniLM-L6-v2` the first time it runs (cached
locally afterward). **This could not be executed in the sandbox this notebook was built in** —
run it on your machine, where you have internet access.


In [20]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(knowledge_base, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

# Quick check: does retrieval pull the right document for an obvious query?
retriever.invoke("why does stormy weather cause delays")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2790.98it/s]


[Document(id='78b3d930-7218-4b74-ad30-e5ce037bb812', metadata={'topic': 'mitigation_traffic_weather'}, page_content='When a delay is caused primarily by Traffic, the recommended customer-facing explanation is to mention congestion on the route and, where relevant, that the rider may be rerouted. When a delay is caused primarily by Weather, the recommended explanation is to mention the specific condition (e.g. heavy rain, storm) and that rider safety takes priority over speed.'),
 Document(id='c546994e-c196-4610-b6f9-f07567e31d54', metadata={'topic': 'weather_definitions'}, page_content='Weather conditions are recorded as Sunny, Cloudy, Windy, Fog, Sandstorms, or Stormy. Sunny weather is treated as the baseline / normal condition. Stormy and Sandstorm conditions are associated with the largest delivery time increases, since they slow down rider travel speed and may require more cautious routing.')]

## Step 5 — Hugging Face LLM ⚠️ requires `huggingface.co` access

`google/flan-t5-base` is used because it's instruction-tuned, runs reasonably on CPU, and
needs no API token or gated-model approval — a good fit for a project that has to run on
whatever machine grades it, not just yours. If your machine is slow, swap in
`google/flan-t5-small` (smaller, faster, slightly less fluent) — same code, just change the
`model_id`.


In [25]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

# Load your GOOGLE_API_KEY from .env
load_dotenv("../.env")
api_key = os.environ.get("GOOGLE_API_KEY", "").strip().strip('"').strip("'")

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    google_api_key=api_key,
    temperature=0.3,
    max_output_tokens=300,
)
print("LLM ready (Google Gemini)")



LLM ready (Google Gemini)


## Step 6 — Grounded Prompt Template

This is the part that keeps the assistant honest: the prompt explicitly instructs the model to
answer **using only** the prediction facts and retrieved context — not to invent its own theory
about why a delivery is slow. The numeric facts come from `predict_delivery_full()`
(Steps 1–2); the policy/definition context comes from the retriever (Step 4).


In [27]:
from langchain_core.prompts import PromptTemplate

ASSISTANT_PROMPT = PromptTemplate.from_template('''\
You are a delivery assistant. Answer the customer's question using ONLY the facts and context
below. Do not invent reasons that are not listed. If the order is not delayed, say so plainly.

Prediction facts:
- Expected delivery time: {expected_time} minutes
- Normal time for this order type: {baseline_time} minutes
- Delayed: {is_delayed}
- Delay amount: {delay_minutes} minutes
- Contributing factors: {reasons}

Relevant policy context:
{context}

Customer question: {question}

Answer:''')


## Step 7 — Full Assistant Function ⚠️ depends on Steps 4–5

Combines everything: run the model, retrieve relevant context, fill the grounded prompt, call
the LLM. This is the single function the API/app layer should call.


In [28]:
def format_reasons(reasons):
    if not reasons:
        return "None -- this order is not delayed."
    return "; ".join(f"{r['feature']} = {r['value']} (+{r['impact_minutes']} min)" for r in reasons)

def ask_assistant(order: dict, question: str) -> str:
    prediction = predict_delivery_full(order)

    retrieved_docs = retriever.invoke(question)
    context = "\n".join(d.page_content for d in retrieved_docs)

    prompt = ASSISTANT_PROMPT.format(
        expected_time=prediction["expected_delivery_time_minutes"],
        baseline_time=prediction["baseline_time_minutes"],
        is_delayed=prediction["is_delayed"],
        delay_minutes=prediction["delay_minutes"],
        reasons=format_reasons(prediction["reasons"]),
        context=context,
        question=question,
    )

    response = llm.invoke(prompt)
    return response.content if hasattr(response, "content") else str(response)




In [29]:
# Example: the same delayed order used throughout 03_Model_Training.ipynb
answer = ask_assistant(test_order, "Why is my order delayed?")
print(answer)


d:\my project\SupplychainX\.venv312\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'Your order is delayed. It has an', 'extras': {'signature': 'EpIJCo8JARFNMg9jfbbkj5hCz8CWpW5yHUqyPLFBrRLAU4XeYdWxARyTzi7Ta5jb3OBdCyVuJNOCc93XYAoXzaLfwa1J4AEsuWKxxVPUoZAvIsVoYwW6g1ELN/x8SqnNipYBY0Zsfv2m3rU9LvoPuVlqtYKA4FLT2h2j+preW6DHqzyLlXI9qJThyBdZbZz39DEluUcDkYARJzxP2sXXv4zbnvsHIjD3E/sQ61vb5tVL5dA7IFq0ZHHiP6fD6HDz3yYOJZOrE3Cfu65jLERyO0Xgg4LuluUAyPq+B+h7rjmu4POLhQwn/JMGosv9VRjQb0OeL+AGN+cqqazMGZlJzO9FfM3Tyn+8T6wjlYL99/1Nohg/qBgM3Jq41NV3PAIDyd5TOQF8XILrxPgWPxNao0xWFNfICJt6vG9q+O3GlKxGzcqFzNjUYtX+mVRedc2+KDMvftWGH3PIqtR85EH1nX/G0DpOFYWlJEKakwxXUlNlCYF0XCJNPMdVRz1cNQIhsgZ02LKMwiz5pgFLzJbMFRUy1rW9cKzn41YaR/H1dt0CuIjZqFA2X+VHJ+egxBHM7qO8dcifB8LEB7WyOG9w97WQ9igPDnxJchM+NGC50MLhasz35hXi9qlwDjiVAhNP88+Fe0f26SiVuqX2iVpE3spgHqK8rrGubttJ+pqWEPNbZS7+qzzQkcGKg6tfNg274QlzJwuQJ2FtKvmsKZlMhxX4ckXx1j3PLPTnHPUMLqwkkisNRkkMUEJG1ppOLyWQGzirR3NyCJCHQ2JPMf1pQyBk1SQnTH/5oWExeeKOInarWP/z1xRKRFKrqbC80iJ/Y24hvHFK8DyJlW0o0r5pTkQFHy8O4L9xTNqq74ElBdwyi/2Ef2AtPWfH4LZByZVYY8FVL5pxzpy2FqYS4

In [30]:
# Example: a non-delayed order
normal_order = {
    "Agent_Age": 30, "Agent_Rating": 4.6, "Distance": 5.0, "Preparation_Time": 8.0,
    "Order_Hour": 11, "Peak_Hour": 0, "Is_Weekend": 0, "Is_Quick_Commerce": 0,
    "Weather": "Sunny", "Traffic": "Low", "Vehicle": "scooter",
    "Area": "Urban", "Category": "Electronics", "Time_of_Day": "Morning",
}
answer = ask_assistant(normal_order, "Is my order going to be late?")
print(answer)


d:\my project\SupplychainX\.venv312\Lib\site-packages\langchain_google_genai\chat_models.py:3237: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'No, your order is not delayed. It is expected', 'extras': {'signature': 'EoIJCv8IARFNMg+jOEG5EbJ0YvvvnknASnYCFpskt77wmhGt6gmOYZVzUbN4CvXj+6HcYXbvkJIYCqWpxTRJ0Rl9+Vafy6cRF2YA5zNJz8iD63vfTDKtttjKKzxFd9ZwxzorfEOWB5DSsswe5mbapGHJP+1POnQJFjOXGxLZ93rvcooFfjJsliGAyXCRIw+lEOIoj/Kf5tN5IegFo6V0+9qBTwJIJgr6sejsfjC/ZMjKovkhteUcXUh36/hZDYHcZZzE1PrVezTaKUujIeQiqYR/EI+N/skor8S5UeWAlfuCM4GDLa+TW1VANCMxO8+CMu4QItGy54oDBtX457jWoxuwYsF0pu1/17w1T7QbIyUrq6o0qhQ4y2Olc95ylrZtBOJuQ/dXb44uVIixi25zIaAuXMJpLULoIMy0K7vZVT7e5gqD1zxd+paQ3wDFglcp5J1pOfJlt6GZkpY6g1fqwINXci1qJNC5OAmoc3Aq5i+GeRXcgb23mHztKFCN2BkaL80zxB8kX92kn2RHor6NJnDvOCIFXEh7HtbAzbIMLK79XbWdP8PBZcuRK1ho73ihllGCaApL3QXsGG9tazSiyT3HPFmmECb2mUgYEr9bujW8u/JB6ZGguYRCoqbCCIXjOf/XZRfxeWr2uwEYpTR9MffvMYg0QRyParFdeSdGYR1cHvFm9ACzX+wonmDBIU4qAt/ZHb5PFHKDdn7LIuF2fZWXZp27thTWTEOCY2xCgcoml4Bj2N5vUTjaN491ApDFMu32yDMUsmahaRzWugXopyfsbQdztdmI3ZAEPWDA6hD8UqCxwWjGid+kHP4VfURUioL6zPvzsPnzrtKrkK6Z6tueGjJxTMslu5SK20HVWuR33uuCabYugSLSpNqpkUe2Dvgw

## Notes for the Report / Next Stage

- **RAG is doing real work here, not decoration:** the retriever supplies policy/definition
  context (e.g. what "Jam" or "Stormy" means, how to phrase a weather-caused delay) that the
  prediction pipeline alone doesn't have — the model outputs numbers, RAG supplies the
  vocabulary and policy to talk about them.
- **Grounding, not generation:** the LLM is deliberately *not* asked to reason about delay
  causes on its own — `predict_delivery_full()` (ML + SHAP) already determined the real causes;
  the LLM's job is only to phrase them naturally, using the retrieved policy context for tone
  and terminology.
- **Model choice is swappable:** `flan-t5-base` was chosen for no-token, reasonable-CPU
  operation. If you have GPU access or an API budget, swapping in a larger instruction-tuned
  model (e.g. `flan-t5-large`, or a hosted API) is a one-line change to Step 5 — nothing else in
  the pipeline needs to change.
- **Next stage:** wrap `ask_assistant()` in the `app/backend`, and connect it to the
  `POST /predict` endpoint your ML module already documents the schema for.
